# Lecția 18 · CNN II, transfer learning

Rar antrenezi o rețea de imagini de la zero. Iei un model deja antrenat pe milioane de imagini (ImageNet) și îl adaptezi la problema ta. Cu date puține, asta e diferența dintre un scor bun și unul jalnic. Modelul a învățat deja să vadă margini, texturi și forme; tu doar îl reorientezi spre clasele tale.

*Celulele de mai jos sunt codul din lecție, în ordine. Unele presupun date pe care le încarci tu (`df`, `X_train`, `y_train`).*

## 01 · De ce funcționează transferul

Un CNN antrenat pe multe imagini învață în straturile de jos trăsături generale: margini, colțuri, texturi. Astea sunt utile pentru aproape orice problemă de imagini, nu doar pentru cea pe care a fost antrenat. Straturile de sus învață trăsături specifice (forme de câini, de mașini). Ideea transferului: păstrezi partea generală și rescrii doar partea specifică.

Așa, în loc să antrenezi milioane de greutăți de la zero, cu milioane de imagini de care nu dispui, folosești ce a învățat deja modelul și antrenezi doar puțin, cu puține date. E cea mai practică tehnică din tot deep learning-ul de imagini.

## 02 · Feature extraction vs fine-tuning

Sunt două moduri de a adapta un model preantrenat, în funcție de cât de multe date ai.

- Feature extraction: îngheți toată rețeaua (nu se mai antrenează) și înlocuiești doar capul de clasificare, pe care îl antrenezi pe clasele tale. Rapid, potrivit când ai foarte puține date.
- Fine-tuning: dezgheți și straturile de sus ale rețelei și le antrenezi cu un learning rate mic, ca să le ajustezi fin spre problema ta. Mai puternic, potrivit când ai date ceva mai multe.

In [ ]:
from torchvision import models
import torch.nn as nn

net = models.resnet18(weights="DEFAULT")
for p in net.parameters():
    p.requires_grad = False          # îngheață tot (feature extraction)
net.fc = nn.Linear(net.fc.in_features, NUM_CLASE)   # cap nou, antrenabil

## 03 · Normalizarea: aceeași cu cea de la antrenare

Un model preantrenat a văzut imagini normalizate într-un anumit fel: scăzute și împărțite cu statisticile ImageNet (mediile și deviațiile pe cele trei canale). Dacă îi dai imagini normalizate altfel, intrarea nu seamănă cu ce a văzut el la antrenare și merge prost, uneori inexplicabil de prost.

> **Reține:** Când folosești un model preantrenat, normalizează imaginile exact cu statisticile pe care a fost antrenat (pentru ImageNet, mediile și deviațiile standard cunoscute). E o greșeală tăcută și frecventă.

## 04 · Learning rate mic la fine-tuning

La fine-tuning, straturile preantrenate au deja greutăți bune. Dacă le antrenezi cu un learning rate mare, pașii mari strică exact trăsăturile valoroase pe care voiai să le păstrezi. Folosește un learning rate mic pentru ele, uneori și mai mic decât pentru capul nou. Așa le ajustezi fin, nu le distrugi.